ENTRENAMIENTO:


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# --- CONFIGURACIÓN ---
ARCHIVO_CIRCULOS = 'circulos.csv'      # Cambia por el nombre real de tu archivo
ARCHIVO_TRIANGULOS = 'triangulos.csv'  # Cambia por el nombre real de tu archivo
UMBRAL_PRESION = 10                    # Cambia esto según el valor que dé tu sensor al presionar
COLUMNAS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz', 'p'] # Ajusta si tus columnas se llaman distinto
COLUMNAS_USAR = ['gx', 'gy', 'gz', 'p']
NORMALIZAR = False

In [ ]:
def normalizar_trazo(trazo):
    """Escala todas las columnas del trazo para que estén entre -1.0 y 1.0"""
    trazo_norm = np.copy(trazo)
    for i in range(trazo.shape[1]): # Iteramos por gx, gy, gz, p
        columna = trazo[:, i]
        min_val = np.min(columna)
        max_val = np.max(columna)
        rango = max_val - min_val

        if rango > 0.0001: # Evitamos dividir por cero si el sensor se quedó plano
            trazo_norm[:, i] = 2.0 * ((columna - min_val) / rango) - 1.0
        else:
            trazo_norm[:, i] = 0.0
    return trazo_norm

def extraer_trazos(df, etiqueta):
    """
    Recorre el dataframe y extrae sub-secuencias continuas donde la presión > UMBRAL_PRESION.
    Devuelve una lista de arrays numpy (los trazos) y una lista de etiquetas.
    """
    trazos = []
    etiquetas = []
    trazo_actual = []

    for _, fila in df.iterrows():
        if fila['p'] > UMBRAL_PRESION:
            # Estamos dibujando: guardamos los 4 valores
            trazo_actual.append(fila[COLUMNAS_USAR].values)
        else:
            # No estamos dibujando
            if len(trazo_actual) > 10:  # Filtramos mini-ruidos (menos de 10 muestras = 200ms)
                if NORMALIZAR:
                    trazo_limpio = normalizar_trazo(np.array(trazo_actual))
                else:
                    trazo_limpio = np.array(trazo_actual)
                trazos.append(trazo_limpio)
                etiquetas.append(etiqueta)
            trazo_actual = [] # Reseteamos para el siguiente trazo

    # Por si el archivo termina mientras se dibujaba
    if len(trazo_actual) > 10:
        if NORMALIZAR:
            trazos.append(normalizar_trazo(np.array(trazo_actual)))
        else:
            trazos.append(np.array(trazo_actual))
        etiquetas.append(etiqueta)

    return trazos, etiquetas

print("1. Cargando y segmentando datos...")
# Cargar CSVs (asegúrate de que no tengan cabeceras raras, o usa skiprows)
df_circulos = pd.read_csv(ARCHIVO_CIRCULOS, names=COLUMNAS, header=0) # header=0 si tienen cabecera
df_triangulos = pd.read_csv(ARCHIVO_TRIANGULOS, names=COLUMNAS, header=0)

# Extraer trazos (0 para círculos, 1 para triángulos)
trazos_circulos, y_circulos = extraer_trazos(df_circulos, 0)
trazos_triangulos, y_triangulos = extraer_trazos(df_triangulos, 1)

X_brutos = trazos_circulos + trazos_triangulos
y = np.array(y_circulos + y_triangulos)

print(f"   -> Encontrados {len(trazos_circulos)} círculos y {len(trazos_triangulos)} triángulos.")

# --- PADDING ---
# Hacemos que todas las secuencias tengan la misma longitud rellenando con un valor especial (-999)
print("2. Acolchando secuencias (Padding)...")
X_padded = pad_sequences(X_brutos, dtype='float32', padding='post', value=-999.0)
print(f"   -> Forma final de los datos X: {X_padded.shape} (muestras, tiempo, sensores)")

# Dividir en entrenamiento (80%) y validación (20%)
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)

# --- CREAR MODELO LSTM ---
print("3. Construyendo modelo LSTM...")
modelo = Sequential()
# Masking le dice a la LSTM que ignore los valores -999.0 que usamos de relleno
modelo.add(Masking(mask_value=-999.0, input_shape=(X_padded.shape[1], X_padded.shape[2])))
modelo.add(LSTM(64, return_sequences=False)) # 64 neuronas de memoria (suficiente para formas)
modelo.add(Dropout(0.5)) # Evita que memorice los datos (overfitting)
modelo.add(Dense(32, activation='relu'))
modelo.add(Dense(1, activation='sigmoid')) # 1 neurona de salida (0=círculo, 1=triángulo)

modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo.summary()

# --- ENTRENAMIENTO
print("\n4. Iniciando entrenamiento...")
historial = modelo.fit(X_train, y_train, epochs=30, batch_size=16, validation_data=(X_test, y_test))

# --- GUARDAR EL MODELO ---
modelo.save('modelo_figuras_lstm.keras')
print("\n¡Modelo guardado como 'modelo_figuras_lstm.h5'!")

1. Cargando y segmentando datos...
   -> Encontrados 124 círculos y 125 triángulos.
2. Acolchando secuencias (Padding)...
   -> Forma final de los datos X: (249, 120, 4) (muestras, tiempo, sensores)
3. Construyendo modelo LSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_1 (Masking)             │ (None, 120, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)


4. Iniciando entrenamiento...
Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 94ms/step - accuracy: 0.4740 - loss: 0.6936 - val_accuracy: 0.6400 - val_loss: 0.6894
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.6728 - loss: 0.6867 - val_accuracy: 0.6000 - val_loss: 0.6767
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.6295 - loss: 0.6799 - val_accuracy: 0.6000 - val_loss: 0.6628
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.5811 - loss: 0.6708 - val_accuracy: 0.6200 - val_loss: 0.6458
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.7320 - loss: 0.6549 - val_accuracy: 0.7800 - val_loss: 0.6260
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.7226 - loss: 0.6250 - val_accuracy: 0.7400 - val_loss: 0.6014
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 96ms/step - accuracy: 0.7369 - loss: 0.6173 - val_accuracy: 0.7600 - val_loss: 0.5735
Epoch 8/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - accuracy: 0.7258 - loss: 0